## Multi-Nodes Slice

In this notebook, we will look at a more customized environment with multiple nodes (still single site), an additional VLAN with custom IP addresses. The individual nodes will have the same custom configuration as well. 

### Configuration

It is perhaps most convenient to have all your customizations specified here. The slice deployment process remains the same throughout, so this initial cell will have all of your configurations set immediately and make it easy to change for different experiment. 

In [ ]:
nodeCnt = 3   # number of nodes
coreCnt = 8   # number of CPUs per node
memSize = 10  # memory per node
diskSize = 50 # storage per node

### Verification

In the next cell, the `fablib` API will verify the token stored in `/home/fabric/.fabric/id_tojen.json` and the slice/bastion keys in `/home/fabric/.ssh/`. 

Information about the specific Fabric project, user ID, and where to find the token and keys are stored in `/home/fabric/.fabric/fabric_rc`. 

In [ ]:
from pathlib import Path
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()
fablib.create_ssh_tunnel_config(overwrite=False)

### Slice Deployment

First, set a unique prefix to your slice. 


In [ ]:
# change the user name to your username
slicePrefix = f"ngo-{nodeCnt}-{coreCnt}-{memSize}-{diskSize}-" 
network_name = "ramnet"
nic_name = "ramnic"

For FABRIC, the availalble resources of each site are presented as total number of cores and aggregated amount of memory. We will recalculate these `minCores` and `minMem` based on our specific request above. To avoid the scenarios where other slices might have slid in and consume resources while we are launching, we will raise these minimums by twice of what are requested. 

In [ ]:
minCores = nodeCnt * coreCnt * 2
minMem = nodeCnt * memSize * 2

We now iterate over the list of available sites, and passover (`continue`) those that don't have adequate resources. 

For those that do, we will attempt to deploy our slice

In [ ]:
from ipaddress import IPv4Network

resources = fablib.get_resources()
resources.update()
sites = resources.get_site_names()

for site in sites:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores < minCores or ram < minMem:
        continue
    sliceName = slicePrefix + site


    # Clean up left-over slices with the same names
    existing = None
    for s in fablib.get_slices():
        if s.get_name() == sliceName:
            print(f"Deleting existing slice: {sliceName}")
            s.delete()
            break

    # Set up new slice
    slice = fablib.new_slice(name=sliceName)
    net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))
    
    for i in range(1, nodeCnt + 1):
        node = slice.add_node(name=f"node{i}",
                              site=site,
                              cores=coreCnt,ram=memSize,disk=diskSize,
                              image="default_ubuntu_22",)
        iface = node.add_component(model="NIC_Basic", name=nic_name).get_interfaces()[0]
        iface.set_mode("config")
        net.add_interface(iface)

    print("==========================================================================")
    try:
        print(f"Submitting slice at {site}...")
        slice.submit(progress=False, wait=True)
    except Exception as e:
        print(f"{siteName}: slice submission failed with error {e}")
        try:
            slice.delete()
        except Exception:
            pass
        continue
            
    # Setup networking
    for i in range(1, nodeCnt + 1):
        node = slice.get_node(name=f"node{i}")
        iface = node.get_interface(network_name=network_name)
        iface.ip_link_up()
        iface.ip_addr_add(
            addr=f"192.168.1.{i}",
            subnet=IPv4Network("192.168.1.0/24"),
        )
        print(f"{node.get_name()} -> 192.168.1.{i}")

    slice.update()
    if slice.get_state() == "Closing":
        print(f"Need to find new site")
        continue
    else: 
        print(f"Slice stable: {slice.isStable()}")
        break

### Slice information

We can print out all relevant information for all the nodes in this slice

In [ ]:
nodes = slice.get_nodes()
for node in nodes:
    print("----", node.get_name(), "----")
    print("management ip:", node.get_management_ip())
    print(node.get_ssh_command())

Let's check the hardware profile of the allocated nodes in our slice

In [ ]:
for node in nodes:
    stdout, stderr = node.execute("cat /proc/cpuinfo | grep processor | wc -l", quiet=True)
    print(stdout)
    stdout, stderr = node.execute("cat /proc/meminfo | grep MemTotal", quiet=True)
    print(stdout)
    stdout, stderr = node.execute("df -h", quiet=True)
    print(stdout)

Checking that the VLAN is working by doing paired ping

In [ ]:
from itertools import permutations

# The second argument (2) specifies the size of the pairs
for pair in permutations(nodes, 2):
    print(pair[0].get_name(), pair[1].get_name())
    des = pair[1].get_interfaces()[0].get_ip_addr()
    stdout, stderr = pair[0].execute(f"ping -c 4 {des}", quiet=True)
    print(stdout)

### Delete Slice

Once complete, make sure to come back and delete your slices, returning resources to the community. 

In [ ]:
# Delete slice
slice.delete()